# 01 - Train an unconditional flow-matching model

Notebook `00` separated the source distribution, learned field, and sampler. We
now train the learned part: a small unconditional MLP that predicts velocity
along independently paired noise-to-data paths. The goal is not merely to lower
a loss curve, but to see whether local velocity predictions assemble into a
convincing global transport. We begin with three clusters, then reuse the same
training idea on a noisy circle and eight separated modes so that curved
support, missing modes, uneven occupancy, and incorrect spread become visible
rather than hidden inside one score.


In [ ]:
import sys
from pathlib import Path

NOTES_DIR = Path.cwd()
if not (NOTES_DIR / "toy_notes.py").exists():
    NOTES_DIR = Path("notebooks/toy_data")
sys.path.insert(0, str(NOTES_DIR.resolve()))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from toy_notes import *
from notebook_views import *

SEED = 2026
set_seed(SEED)
device = choose_device()
print("device:", device)


## 1. Why train the model ourselves?

A pretrained checkpoint would hide the connection between sampled endpoint
pairs and the velocity loss. This small example lets us see the complete
training rule before using larger pretrained image models.

For each minibatch:

1. sample independent `x_noise` and `x_data`;
2. sample `t` uniformly from `[0,1]`;
3. form `x_t = t*x_data + (1-t)*x_noise`;
4. regress `v_theta(t,x_t)` onto `x_data - x_noise` with mean-squared error.

This is I-CFM. It is not optimal-transport CFM because we do not solve an
optimal coupling between noise and data samples.


In [ ]:
def train_visible_flow(model, steps):
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-5)
    losses = []
    model.train()

    for step in range(steps):
        x_data, _ = sample_labeled_mixture(1024, device=device)
        x_noise = sample_base(1024, device=device)
        t = torch.rand(1024, device=device)
        x_t = (1 - t[:, None]) * x_noise + t[:, None] * x_data
        target_velocity = x_data - x_noise

        loss = (model(t, x_t) - target_velocity).square().mean()
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())

    model.eval()
    return losses

model, losses, trained_now = load_or_train_model(device=device, trainer=train_visible_flow)
print("trained now:", trained_now, "| parameters:", sum(p.numel() for p in model.parameters()))


In [ ]:
plot_loss_curve(losses, title="Flow-matching training loss")


## 2. How does a fitted field become samples?

Once an initial noise batch is fixed, the ODE sampler is deterministic:

$$\frac{dx}{dt}=v_\theta(t,x), \qquad x(0)=x_{noise}.$$

We use Heun's method here. Notebook `02` compares it with Euler and RK4.


In [ ]:
set_seed(SEED + 10)
x0_eval = sample_base(3000, device=device)
trajectory, sample_seconds = timed_sample(base_velocity(model), x0_eval, steps=80, method="heun")
generated = trajectory[-1].to(device)
target, target_labels = sample_labeled_mixture(3000, device=device)

plot_flow_endpoints(x0_eval, target, target_labels, generated)
print(f"sampling time: {sample_seconds:.3f} s")


In [ ]:
diagnostics = pd.DataFrame(flow_endpoint_diagnostics(generated, target, target_labels))
display(diagnostics.round(4))


### Why use Wasserstein distance in addition to a plot?

A class rate or a mean checks only one property. Wasserstein distance asks for
the cheapest way to match generated points to target points, where moving a
point farther costs more. The code uses equal-size deterministic subsets and
reports the mean matched distance. It is a finite-sample estimate, not the
unknown population distance.

The lines below make the definition literal. Lower is better. We still inspect
mode coverage and spread because one scalar cannot diagnose every failure.


In [ ]:
_ = plot_wasserstein_matching(generated, target)


In [ ]:
plot_velocity_field(model, device)


## 3. Does the method learn a curved distribution?

Three Gaussian clusters could leave the impression that flow matching only
learns a few destination centers. A noisy circle is a stricter visual test: it
has continuously many valid endpoints and a thin curved support. We train a
separate unconditional model and ask whether it forms the ring without leaving
large angular gaps.


In [ ]:
circle = run_flow_case(
    load_or_train_circle_model, sample_noisy_circle,
    device=device, seed=SEED + 71,
)


In [ ]:
plot_circle_flow(circle)


In [ ]:
display(pd.DataFrame(circle_flow_metrics(circle)).round(4))


## 4. Why also inspect eight separated modes?

The three-class model is kept for the steering notebooks because its labels are
easy to discuss, while the circle tests continuous curved geometry. Eight narrow
components expose a different failure mode: a model can omit destinations or
assign them the wrong amount of probability. This is still learned I-CFM; no
destination center is used by the sampler.

We color each particle by the component nearest to its **final** endpoint, then
trace the same particle backward through stored time slices. The colors are for
reading the map after sampling; the model never receives them.


In [ ]:
balanced_labels = torch.arange(8, device=device).repeat_interleave(512)
eight_modes = run_flow_case(
    load_or_train_eight_gaussian_model,
    sample_eight_gaussians,
    device=device,
    seed=SEED + 81,
    sampler_kwargs={"labels": balanced_labels},
)


In [ ]:
plot_eight_gaussian_flow(eight_modes)


In [ ]:
eight_summary, per_mode = eight_gaussian_metrics(eight_modes)
display(pd.DataFrame(per_mode).round(3))
display(pd.DataFrame(eight_summary).round(4))


The snapshot figure shows **where the learned ODE sends regions of the initial
Gaussian**, while the diagnostics describe mode coverage, occupancy, and
within-mode spread. No single plot or scalar captures every mismatch between
the generated and target distributions.


## From a trained field to a denoised estimate

**Questions**

1. Why can a low minibatch MSE coexist with visibly poor samples?
2. What information about class identity did the unconditional model receive?
3. At which time does the vector field mostly choose coarse destination modes?

These examples show why training loss is only the beginning of the story. The
circle and eight-mode models are diagnostic detours: they reveal
curved-support and occupancy failures, while the remaining toy notes return to
the three-cluster model for class-level steering. With one trained field in
hand, Notebook `02` asks how to read its prediction as a denoised estimate and
how faithfully different deterministic solvers follow the same field.
